# Notebook 01 — Geração de Dados Sintéticos

## Objetivo

Gerar dados sintéticos para um projeto de BI de E-commerce. Serão criadas três tabelas fonte:
- **clientes**: 1.500 registros com dados demográficos
- **produtos**: 250 registros com catálogo de produtos
- **pedidos**: 8.000 registros com transações de vendas

Os dados serão salvos em formato Parquet no diretório `data/bronze/` para serem consumidos pela camada Bronze do pipeline Medallion.


## 1. Importação de Bibliotecas

Utilizaremos `pandas` e `numpy` para geração dos dados, `random` e `datetime` para valores aleatórios realistas, e `pyarrow` para salvar os arquivos Parquet.


In [ ]:
import os
import random
import datetime
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

random.seed(42)
np.random.seed(42)

print("Bibliotecas importadas com sucesso.")


## 2. Constantes do Projeto

Definimos o número de registros de cada tabela e o caminho de saída.


In [ ]:
N_CLIENTES = 1500
N_PRODUTOS = 250
N_PEDIDOS = 8000
N_VENDEDORES = 15

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "data", "bronze")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Diretório de saída: {OUTPUT_DIR}")


## 3. Listas Auxiliares

Criamos listas de nomes, cidades, estados, categorias e nomes de vendedores para compor os dados sintéticos.


In [ ]:
NOMES = [
    "Ana Silva", "Carlos Souza", "Mariana Oliveira", "João Santos", "Pedro Lima",
    "Fernanda Costa", "Rafael Pereira", "Juliana Almeida", "Lucas Ferreira", "Patrícia Rodrigues",
    "Bruno Carvalho", "Camila Nunes", "Daniel Barbosa", "Gabriela Ribeiro", "Felipe Martins",
    "Amanda Teixeira", "André Moreira", "Larissa Cardoso", "Marcos Azevedo", "Beatriz Gomes",
    "Thiago Araújo", "Vanessa Melo", "Ricardo Pinto", "Isabela Correia", "Eduardo Freitas",
    "Tatiane Rocha", "Sérgio Neves", "Débora Mendes", "Roberto Dias", "Natália Vieira",
    "Paulo Cunha", "Elaine Monteiro", "Fábio Campos", "Carla Santana", "Renato Nascimento",
    "Aline Farias", "Gustavo Peixoto", "Cristina Araújo", "Marcelo Moura", "Bianca Duarte",
    "Alexandre Machado", "Priscila Tavares", "Leonardo Rezende", "Simone Andrade", "Vitor Pires",
    "Adriana Lopes", "Henrique Braga", "Mônica Siqueira", "César Moraes", "Sabrina Macedo"
]

CIDADES = [
    "São Paulo", "Rio de Janeiro", "Belo Horizonte", "Curitiba", "Porto Alegre",
    "Salvador", "Recife", "Fortaleza", "Brasília", "Manaus",
    "Belém", "Goiânia", "São Luís", "Maceió", "Natal",
    "Campo Grande", "Teresina", "João Pessoa", "Aracaju", "Vitória",
    "Florianópolis", "Cuiabá", "Uberlândia", "Ribeirão Preto", "Santos"
]

ESTADOS_MAP = {
    "São Paulo": ("SP", "Sudeste"), "Rio de Janeiro": ("RJ", "Sudeste"),
    "Belo Horizonte": ("MG", "Sudeste"), "Curitiba": ("PR", "Sul"),
    "Porto Alegre": ("RS", "Sul"), "Salvador": ("BA", "Nordeste"),
    "Recife": ("PE", "Nordeste"), "Fortaleza": ("CE", "Nordeste"),
    "Brasília": ("DF", "Centro-Oeste"), "Manaus": ("AM", "Norte"),
    "Belém": ("PA", "Norte"), "Goiânia": ("GO", "Centro-Oeste"),
    "São Luís": ("MA", "Nordeste"), "Maceió": ("AL", "Nordeste"),
    "Natal": ("RN", "Nordeste"), "Campo Grande": ("MS", "Centro-Oeste"),
    "Teresina": ("PI", "Nordeste"), "João Pessoa": ("PB", "Nordeste"),
    "Aracaju": ("SE", "Nordeste"), "Vitória": ("ES", "Sudeste"),
    "Florianópolis": ("SC", "Sul"), "Cuiabá": ("MT", "Centro-Oeste"),
    "Uberlândia": ("MG", "Sudeste"), "Ribeirão Preto": ("SP", "Sudeste"),
    "Santos": ("SP", "Sudeste")
}

CATEGORIAS_PRODUTOS = [
    ("Eletrônicos", "Smartphones"), ("Eletrônicos", "Notebooks"),
    ("Eletrônicos", "Tablets"), ("Eletrônicos", "Fones de Ouvido"),
    ("Eletrônicos", "Monitores"), ("Vestuário", "Camisetas"),
    ("Vestuário", "Calças"), ("Vestuário", "Jaquetas"),
    ("Vestuário", "Tênis"), ("Vestuário", "Acessórios"),
    ("Casa", "Decoração"), ("Casa", "Utensílios"),
    ("Casa", "Móveis"), ("Casa", "Iluminação"),
    ("Livros", "Ficção"), ("Livros", "Não Ficção"),
    ("Livros", "Infantil"), ("Livros", "Técnicos"),
    ("Esportes", "Fitness"), ("Esportes", "Camping"),
    ("Esportes", "Ciclismo"), ("Esportes", "Natação"),
    ("Brinquedos", "Jogos de Tabuleiro"), ("Brinquedos", "Bonecos"),
    ("Brinquedos", "Eletrônicos Infantis")
]

NOMES_VENDEDORES = [
    "Roberto Alves", "Carla Mendes", "Márcio Teixeira", "Patrícia Lopes", "José Barros",
    "Sandra Matos", "Antônio Rangel", "Denise Quadros", "Jorge Fonseca", "Lúcia Vianna",
    "Wilson Borges", "Márcia Leal", "Hélio Prado", "Vera Coelho", "Nelson Guará"
]

print(f"{len(NOMES)} nomes, {len(CIDADES)} cidades, {len(CATEGORIAS_PRODUTOS)} subcategorias, {len(NOMES_VENDEDORES)} vendedores")


## 4. Geração da Tabela `clientes`

Geramos 1.500 clientes com dados realistas: nome, email, telefone, cidade, estado, região e data de cadastro.


In [ ]:
def gerar_telefone():
    ddd = random.randint(11, 99)
    parte1 = random.randint(90000, 99999)
    parte2 = random.randint(1000, 9999)
    return f"({ddd}) 9{parte1}-{parte2}"

clientes_data = []
for i in range(1, N_CLIENTES + 1):
    nome = random.choice(NOMES)
    email_nome = nome.lower().replace(" ", ".")
    email_prov = random.choice(["gmail.com", "hotmail.com", "outlook.com", "yahoo.com.br"])
    email = f"{email_nome}{random.randint(1, 99)}@{email_prov}"
    telefone = gerar_telefone()
    cidade = random.choice(CIDADES)
    estado, regiao = ESTADOS_MAP[cidade]
    data_cadastro = datetime.date(2021, 1, 1) + datetime.timedelta(days=random.randint(0, 1095))
    clientes_data.append({
        "cliente_id": i,
        "nome": nome,
        "email": email,
        "telefone": telefone,
        "cidade": cidade,
        "estado": estado,
        "regiao": regiao,
        "data_cadastro": data_cadastro
    })

df_clientes = pd.DataFrame(clientes_data)
print(f"clientes: {len(df_clientes)} registros gerados")


## 5. Geração da Tabela `produtos`

Geramos 250 produtos com SKU, nome, categoria, subcategoria, preços e estoque.


In [ ]:
PRODUTOS_NOMES_POR_CAT = {
    ("Eletrônicos", "Smartphones"): ["Galaxy S", "iPhone Pro", "Moto G", "Xiaomi Note", "ZenFone"],
    ("Eletrônicos", "Notebooks"): ["ThinkPad", "MacBook Air", "Inspiron", "Pavilion", "Aspire"],
    ("Eletrônicos", "Tablets"): ["iPad", "Galaxy Tab", "Kindle Fire", "Mi Pad", "Tab"],
    ("Eletrônicos", "Fones de Ouvido"): ["AirPods", "Galaxy Buds", "Headset BT", "Earbuds Pro", "Studio"],
    ("Eletrônicos", "Monitores"): ["UltraSharp", "Pro Display", "Gamer View", "Curvo WQHD", "Full HD LED"],
    ("Vestuário", "Camisetas"): ["Camiseta Basic", "Camiseta Estampa", "Camiseta Polo", "Regata Sport", "Camiseta Oversized"],
    ("Vestuário", "Calças"): ["Jeans Slim", "Sarja Reta", "Jogger Cargo", "Chino Slim", "Bermuda Jeans"],
    ("Vestuário", "Jaquetas"): ["Jaqueta Jeans", "Corta Vento", "Bomber Premium", "Parka Inverno", "Moletom Fechado"],
    ("Vestuário", "Tênis"): ["Tênis Esportivo", "Tênis Casual", "Running Pro", "Skate Shoe", "Slip-On Canvas"],
    ("Vestuário", "Acessórios"): ["Boné Snapback", "Cinto Couro", "Óculos Sol", "Relógio Digital", "Mochila Compacta"],
    ("Casa", "Decoração"): ["Vaso Cerâmica", "Quadro Decorativo", "Almofada Veludo", "Cortina Blackout", "Tapete Felpudo"],
    ("Casa", "Utensílios"): ["Jogo de Panelas", "Faqueiro Inox", "Jogo de Copos", "Assadeira Antiaderente", "Utensílios Silicone"],
    ("Casa", "Móveis"): ["Mesa de Centro", "Estante Modular", "Cadeira Ergonômica", "Sofá Retrátil", "Rack para TV"],
    ("Casa", "Iluminação"): ["Luminária Pendente", "Abajur Articulado", "Fita LED RGB", "Lustre Cristal", "Luminária de Mesa"],
    ("Livros", "Ficção"): ["O Mistério do Lago", "A Última Fronteira", "Sombras do Passado", "Contos do Amanhã", "Trilogia Elemental"],
    ("Livros", "Não Ficção"): ["Economia Prática", "História do Brasil", "Mindset de Sucesso", "Ciência Revelada", "Guia do Investidor"],
    ("Livros", "Infantil"): ["A Turma na Floresta", "Pequeno Explorador", "Fadas e Dragões", "Bichos Amigos", "Nave Espacial"],
    ("Livros", "Técnicos"): ["Python Avançado", "Redes de Computadores", "Estatística Aplicada", "Engenharia de Dados", "Machine Learning Prático"],
    ("Esportes", "Fitness"): ["Halteres Emborrachados", "Tapete Yoga", "Corda de Pular", "Faixa Elástica", "Bola de Pilates"],
    ("Esportes", "Camping"): ["Barraca 4 Pessoas", "Saco de Dormir", "Lanterna Recarregável", "Fogareiro Portátil", "Mochila Trilha"],
    ("Esportes", "Ciclismo"): ["Bicicleta MTB", "Capacete Ventilado", "Luvas Gel", "Cadeirinha Bike", "Kit Farol LED"],
    ("Esportes", "Natação"): ["Óculos Natação", "Touca Silicone", "Maiô Treino", "Prancha Flutuação", "Nadadeira Curta"],
    ("Brinquedos", "Jogos de Tabuleiro"): ["War Edição Especial", "Catan", "Dixit", "Ticket to Ride", "Carcassonne"],
    ("Brinquedos", "Bonecos"): ["Boneca Fashion", "Action Figure Hero", "Kit Dinossauros", "Boneco Super-Herói", "Família Sylvanian"],
    ("Brinquedos", "Eletrônicos Infantis"): ["Tablet Infantil", "Carrinho Controle", "Drone Mini", "Robô Programável", "Karaokê Infantil"]
}

produtos_data = []
sku_counter = 1000
for (categoria, subcategoria), nomes in PRODUTOS_NOMES_POR_CAT.items():
    for i, nome_base in enumerate(nomes):
        sku = f"SKU-{sku_counter:04d}"
        sku_counter += 1
        preco_custo = round(random.uniform(15.0, 800.0), 2)
        preco_venda = round(preco_custo * random.uniform(1.3, 2.5), 2)
        estoque = random.randint(0, 500)
        produtos_data.append({
            "sku": sku,
            "nome_produto": f"{nome_base} {random.choice(['Plus', 'Pro', 'Lite', 'Max', 'Ultra', ''])}".strip(),
            "categoria": categoria,
            "subcategoria": subcategoria,
            "preco_venda": preco_venda,
            "preco_custo": preco_custo,
            "estoque": estoque
        })

df_produtos = pd.DataFrame(produtos_data)
df_produtos = df_produtos.iloc[:N_PRODUTOS]
print(f"produtos: {len(df_produtos)} registros gerados")


## 6. Geração da Tabela `pedidos`

Geramos 8.000 pedidos com integridade referencial: cada pedido referencia um `cliente_id` válido e um `sku` válido.


In [ ]:
cliente_ids = df_clientes["cliente_id"].tolist()
sku_list = df_produtos["sku"].tolist()
cidades_list = df_clientes["cidade"].tolist()

pedidos_data = []
for i in range(1, N_PEDIDOS + 1):
    cliente_id = random.choice(cliente_ids)
    sku = random.choice(sku_list)
    data_pedido = datetime.date(2023, 1, 1) + datetime.timedelta(days=random.randint(0, 700))
    quantidade = random.randint(1, 5)
    valor_frete = round(random.uniform(8.0, 45.0), 2)
    cidade = random.choice(CIDADES)
    estado, regiao = ESTADOS_MAP[cidade]
    id_vendedor = random.randint(1, N_VENDEDORES)
    pedidos_data.append({
        "pedido_id": i,
        "cliente_id": cliente_id,
        "sku": sku,
        "data_pedido": data_pedido,
        "quantidade": quantidade,
        "valor_frete": valor_frete,
        "municipio": cidade,
        "estado": estado,
        "regiao": regiao,
        "id_vendedor": id_vendedor
    })

df_pedidos = pd.DataFrame(pedidos_data)
print(f"pedidos: {len(df_pedidos)} registros gerados")


## 7. Amostras dos Dados Gerados

Exibimos as primeiras linhas de cada tabela para inspeção visual.


In [ ]:
print("=" * 60)
print("TABELA: clientes")
print("=" * 60)
display(df_clientes.head(5))
print(f"\nShape: {df_clientes.shape}")

print("\n" + "=" * 60)
print("TABELA: produtos")
print("=" * 60)
display(df_produtos.head(5))
print(f"\nShape: {df_produtos.shape}")

print("\n" + "=" * 60)
print("TABELA: pedidos")
print("=" * 60)
display(df_pedidos.head(5))
print(f"\nShape: {df_pedidos.shape}")


## 8. Salvando em Formato Parquet

Cada DataFrame é salvo como arquivo Parquet no diretório `data/bronze/`.


In [ ]:
df_clientes.to_parquet(os.path.join(OUTPUT_DIR, "clientes.parquet"), index=False)
print(f"[OK] clientes.parquet salvo em {OUTPUT_DIR}")

df_produtos.to_parquet(os.path.join(OUTPUT_DIR, "produtos.parquet"), index=False)
print(f"[OK] produtos.parquet salvo em {OUTPUT_DIR}")

df_pedidos.to_parquet(os.path.join(OUTPUT_DIR, "pedidos.parquet"), index=False)
print(f"[OK] pedidos.parquet salvo em {OUTPUT_DIR}")


## 9. Estatísticas Resumidas

Exibimos um resumo dos dados gerados: contagem de linhas, período coberto, estimativa de receita.


In [ ]:
print("=" * 50)
print("RESUMO DOS DADOS GERADOS")
print("=" * 50)
print(f"Clientes gerados:      {len(df_clientes):>5}")
print(f"Produtos gerados:      {len(df_produtos):>5}")
print(f"Pedidos gerados:       {len(df_pedidos):>5}")
print(f"Vendedores distintos:  {df_pedidos['id_vendedor'].nunique():>5}")
print(f"\nPeríodo dos pedidos: {df_pedidos['data_pedido'].min()} a {df_pedidos['data_pedido'].max()}")

df_pedidos_merged = df_pedidos.merge(df_produtos[["sku", "preco_venda"]], on="sku", how="left")
df_pedidos_merged["total_estimado"] = df_pedidos_merged["quantidade"] * df_pedidos_merged["preco_venda"]
receita_estimada = df_pedidos_merged["total_estimado"].sum()
frete_total = df_pedidos["valor_frete"].sum()

print(f"\nReceita bruta estimada: R$ {receita_estimada:,.2f}")
print(f"Frete total cobrado:    R$ {frete_total:,.2f}")
print(f"Ticket médio estimado:  R$ {receita_estimada / N_PEDIDOS:,.2f}")

print(f"\nRegiões nos pedidos:")
for regiao, count in df_pedidos["regiao"].value_counts().items():
    print(f"  {regiao}: {count} pedidos ({count/N_PEDIDOS*100:.1f}%)")


## Conclusão

Os dados sintéticos foram gerados com sucesso e salvos em Parquet. As três tabelas fonte estão prontas para serem ingeridas na **Camada Bronze** do pipeline Medallion.

- **clientes**: 1.500 registros com dados demográficos
- **produtos**: 250 produtos em 6 categorias e 25 subcategorias  
- **pedidos**: 8.000 transações com FK íntegras e período de ~2 anos

No próximo notebook (**NB02**), faremos a ingestão desses dados para a camada Bronze utilizando PySpark + Delta Lake.
